In [1]:
from pathlib import Path

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)

from langchain_core.documents import Document

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append("../")

from src.loaders.document_loader import EnterpriseDocumentLoader

DATASET = "../datasets/military"

loader = EnterpriseDocumentLoader()

documents = loader.load_directory(DATASET)

print(f"Loaded Documents : {len(documents)}")

Loaded Documents : 18


In [3]:
character_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=100
)

character_chunks = character_splitter.split_documents(documents)

print(len(character_chunks))

25


In [4]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

recursive_chunks = recursive_splitter.split_documents(documents)

print(len(recursive_chunks))

29


In [5]:
chunk = recursive_chunks[0]

print(chunk.page_content)

print()

print(chunk.metadata)

OPERATOR:           "Spectre-7" (Sgt. Marcus Webb)
MISSION:            Op: BROKEN HAMMER
DATE:               [REDACTED]
LOCATION:           Verdansk, Kastovia
UNIT:               Demon Dogs, 1st Platoon

{'source': '..\\datasets\\military\\Mission_Log.txt', 'filename': 'Mission_Log.txt', 'extension': '.txt', 'domain': 'military'}


In [6]:
for i, chunk in enumerate(recursive_chunks):

    chunk.metadata["chunk_id"] = i
    chunk.metadata["chunk_size"] = len(chunk.page_content)

In [7]:
sizes = [
    chunk.metadata["chunk_size"]
    for chunk in recursive_chunks
]

print("Total Chunks :", len(sizes))

print("Average Size :", sum(sizes)/len(sizes))

print("Minimum :", min(sizes))

print("Maximum :", max(sizes))

Total Chunks : 29
Average Size : 308.7586206896552
Minimum : 207
Maximum : 496


In [8]:
for chunk in recursive_chunks[:5]:

    print("="*70)

    print(chunk.metadata)

    print()

    print(chunk.page_content[:300])

{'source': '..\\datasets\\military\\Mission_Log.txt', 'filename': 'Mission_Log.txt', 'extension': '.txt', 'domain': 'military', 'chunk_id': 0, 'chunk_size': 336}

OPERATOR:           "Spectre-7" (Sgt. Marcus Webb)
MISSION:            Op: BROKEN HAMMER
DATE:               [REDACTED]
LOCATION:           Verdansk, Kastovia
UNIT:               Demon Dogs, 1st Platoon
{'source': '..\\datasets\\military\\Mission_Log.txt', 'filename': 'Mission_Log.txt', 'extension': '.txt', 'domain': 'military', 'chunk_id': 1, 'chunk_size': 472}

--- 0430 HOURS ---
Insertion via HALO jump successful. LZ cold. Full squad accounted 
for: Myself, "Mother" (Jenkins), "Prophet" (Okonkwo), "Grinch" 
(Mackie). Moving to OP Alpha. Comms check green across board.

--- 0515 HOURS ---
Contact. Small arms fire from the hospital complex. Prophet 
confirm
{'source': '..\\datasets\\military\\Mission_Log.txt', 'filename': 'Mission_Log.txt', 'extension': '.txt', 'domain': 'military', 'chunk_id': 2, 'chunk_size': 253}

Prophet

In [9]:
from src.splitters.chunker import EnterpriseChunker

chunker = EnterpriseChunker(
    chunk_size=500,
    overlap=100,
    strategy="recursive"
)

chunks = chunker.split_documents(documents)

print("Chunks :", len(chunks))

Chunks : 29


In [10]:
chunks[10]

Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 10, 'chunk_size': 496}, page_content='Killstreak  Active  (Flashing  Red  Square)  [ ◆ ]  -  High  Value  Target  (Gold  Diamond  -  Priority  Elimination)   SECTION  3:  COUNTER-DETECTION  -----------------------------  WARNING:  Enemy  operators  may  deploy  the  following  countermeasures:  -  CUAV  "Scrambler"  Drone:  Degrades  radar  resolution  by  60%  -  EMP  Field  Generator:  Complete  system  blackout  for  15-30  seconds  -  Ghost  Perk  Operatives:  Invisible  to  standard  sweeps  -  Cold-Blooded  Operatives:  No')

In [11]:
import pickle

with open("../vector_store/chunks.pkl","wb") as f:

    pickle.dump(chunks,f)

print("Chunks Saved")

Chunks Saved
